# ⚡ 03. Expanding-Window Backtesting Protocol

**Protocol**: 5-Fold expanding window covering the final $5 \times 168 = 840$ hours.

In [ ]:
from dataclasses import dataclass
from typing import Iterator
import pandas as pd

@dataclass
class BacktestFold:
    fold_id: int
    train_start: pd.Timestamp
    train_end: pd.Timestamp
    test_start: pd.Timestamp
    test_end: pd.Timestamp

def expanding_window_folds(
    df: pd.DataFrame,
    n_folds: int = 5,
    horizon_hours: int = 168,
    gap_hours: int = 0,
) -> Iterator[BacktestFold]:
    total_hours = len(df)
    test_pool = n_folds * horizon_hours
    test_region_start_idx = total_hours - test_pool
    
    for fold in range(n_folds):
        test_start_idx = test_region_start_idx + fold * horizon_hours
        test_end_idx   = test_start_idx + horizon_hours
        train_end_idx  = test_start_idx - gap_hours - 1
        
        yield BacktestFold(
            fold_id     = fold + 1,
            train_start = df.index[0],
            train_end   = df.index[train_end_idx],
            test_start  = df.index[test_start_idx],
            test_end    = df.index[test_end_idx - 1],
        )

# Example fold extraction
df = pd.read_csv("PJME_hourly.csv", parse_dates=['Datetime'], index_col='Datetime').sort_index()
for f in expanding_window_folds(df, n_folds=5, horizon_hours=168):
    print(f"Fold {f.fold_id}: Train [{f.train_start.date()} to {f.train_end.date()}] -> Test [{f.test_start} to {f.test_end}]")
